# Adım 3: Spark Structured Streaming ve Delta Lake

Bu adımda Kafka üzerinden gelen gerçek zamanlı JSON mesajları Apache Spark Structured Streaming ile okunmuştur. Gelen veriler parse edilmiş, null ve duplicate kayıtlar temizlenmiş, ardından Bronze/Silver/Gold mimarisiyle Delta Lake formatında kaydedilmiştir.

## Kullanılan Kafka Bilgileri

- Kafka broker: `kafka:9092`
- Kafka topic: `project-topic`
- Mesaj formatı: JSON
- Veri kaynağı: `World Energy Consumption` veri seti


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, get_json_object, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = SparkSession.builder \
    .appName("WorldEnergyKafkaToDelta") \
    .config("spark.sql.shuffle.partitions", "2") \
    .config("spark.hadoop.hadoop.security.authentication", "simple") \
    .getOrCreate()

outer_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("event_type", StringType(), True),
    StructField("related_id", IntegerType(), True),
    StructField("data", StringType(), True)
])

raw_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "project-topic") \
    .option("startingOffsets", "latest") \
    .load()

bronze_df = raw_df.selectExpr("CAST(value AS STRING) as raw_message") \
    .withColumn("ingestion_time", current_timestamp())

parsed_df = bronze_df.withColumn(
    "json_data",
    from_json(col("raw_message"), outer_schema)
)

silver_df = parsed_df.select(
    col("json_data.timestamp").alias("event_timestamp"),
    col("json_data.user_id").alias("user_id"),
    col("json_data.event_type").alias("event_type"),
    col("json_data.related_id").alias("related_id"),

    get_json_object(col("raw_message"), "$.data.country").alias("country"),
    get_json_object(col("raw_message"), "$.data.year").cast("int").alias("year"),
    get_json_object(col("raw_message"), "$.data.population").cast("double").alias("population"),
    get_json_object(col("raw_message"), "$.data.gdp").cast("double").alias("gdp"),
    get_json_object(col("raw_message"), "$.data.energy_per_capita").cast("double").alias("energy_per_capita"),
    get_json_object(col("raw_message"), "$.data.energy_per_gdp").cast("double").alias("energy_per_gdp"),
    get_json_object(col("raw_message"), "$.data.electricity_generation").cast("double").alias("electricity_generation"),
    get_json_object(col("raw_message"), "$.data.renewables_electricity").cast("double").alias("renewables_electricity"),
    get_json_object(col("raw_message"), "$.data.fossil_electricity").cast("double").alias("fossil_electricity"),
    get_json_object(col("raw_message"), "$.data.carbon_intensity_elec").cast("double").alias("carbon_intensity_elec"),

    col("ingestion_time")
)

# Null temizleme
silver_df = silver_df.na.drop(subset=[
    "country",
    "year",
    "population",
    "gdp",
    "energy_per_capita",
    "electricity_generation",
    "renewables_electricity",
    "fossil_electricity",
    "carbon_intensity_elec"
])

# Duplicate temizleme
silver_df = silver_df.dropDuplicates([
    "country",
    "year",
    "event_type",
    "related_id"
])

gold_df = silver_df \
    .withColumn(
        "renewable_ratio",
        col("renewables_electricity") / col("electricity_generation")
    ) \
    .withColumn(
        "fossil_ratio",
        col("fossil_electricity") / col("electricity_generation")
    ) \
    .withColumn(
        "gdp_per_capita",
        col("gdp") / col("population")
    ) \
    .withColumn(
        "energy_efficiency_score",
        col("gdp") / col("energy_per_capita")
    ) \
    .withColumn(
        "carbon_risk_score",
        col("carbon_intensity_elec") * col("fossil_ratio")
    )

bronze_query = bronze_df.writeStream \
    .format("delta") \
    .option("path", "/data/bronze_energy") \
    .option("checkpointLocation", "/data/checkpoints/bronze") \
    .outputMode("append") \
    .start()

silver_query = silver_df.writeStream \
    .format("delta") \
    .option("path", "/data/silver_energy") \
    .option("checkpointLocation", "/data/checkpoints/silver") \
    .outputMode("append") \
    .start()

gold_query = gold_df.writeStream \
    .format("delta") \
    .option("path", "/data/gold_energy") \
    .option("checkpointLocation", "/data/checkpoints/gold") \
    .outputMode("append") \
    .start()

gold_console = gold_df.writeStream \
    .format("console") \
    .option("truncate", "false") \
    .outputMode("append") \
    .start()

spark.streams.awaitAnyTermination()


## Veri Temizleme

Bu aşamada JSON mesajları kolonlara ayrılmıştır. Kritik kolonlarda null değer içeren kayıtlar temizlenmiştir. Ayrıca aynı ülke, yıl, event_type ve related_id bilgilerine sahip tekrar eden kayıtlar duplicate kabul edilerek kaldırılmıştır.


## Üretilen Feature'lar

Gold katmanında 5 yeni feature üretilmiştir:

1. `renewable_ratio`: Yenilenebilir elektrik üretiminin toplam elektrik üretimine oranı
2. `fossil_ratio`: Fosil kaynaklı elektrik üretiminin toplam elektrik üretimine oranı
3. `gdp_per_capita`: Kişi başına düşen GDP
4. `energy_efficiency_score`: GDP ve enerji tüketimi ilişkisine göre verimlilik skoru
5. `carbon_risk_score`: Karbon yoğunluğu ve fosil enerji oranına göre risk skoru


## Delta Lake Kontrolü

Bronze, Silver ve Gold klasörleri kontrol edilmiştir. Her klasörde `_delta_log` dizininin oluşması, verilerin Delta Lake formatında başarılı şekilde yazıldığını göstermektedir.

Kontrol edilen dizinler:

- `/data/bronze_energy`
- `/data/silver_energy`
- `/data/gold_energy`


## Sonuç

Bu adım sonunda Kafka → Spark Structured Streaming → Delta Lake akışı başarıyla kurulmuştur. Veri gerçek zamanlı olarak okunmuş, temizlenmiş, özellik mühendisliği uygulanmış ve Delta Lake Bronze/Silver/Gold mimarisinde saklanmıştır.
